# Advanced 07 — Discrete-Time Markov Chains

Practice notebook for the **"Discrete-Time Markov Chains"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — The Markov property and a transition matrix

A sequence \( (X_n)_{n\geq 0} \) on a countable state space \( S \) is a
**Markov chain** with transition probabilities \( P=(p_{ij}) \) if

$$
P(X_{n+1}=j \mid X_n=i,\, X_{n-1}=i_{n-1},\dots,X_0=i_0)
= P(X_{n+1}=j \mid X_n=i) = p_{ij}.
$$

This is the **Markov property**: the future depends on the past only through the
present. Each row of \( P \) is a probability distribution:
\( p_{ij}\geq 0 \) and \( \sum_j p_{ij}=1 \).

**Example.** A small 3-state weather model on
\( S=\{\text{Sunny},\text{Cloudy},\text{Rainy}\} \). We define \( P \),
check it is a valid stochastic matrix, and simulate one path by drawing successive
states from the row corresponding to the current state.


In [ ]:
# 3-state weather chain: 0=Sunny, 1=Cloudy, 2=Rainy
P = np.array([
    [0.70, 0.20, 0.10],   # from Sunny
    [0.30, 0.40, 0.30],   # from Cloudy
    [0.20, 0.30, 0.50],   # from Rainy
])
state_names = ["Sunny", "Cloudy", "Rainy"]

# Sanity check: rows are probability distributions
assert P.shape == (3, 3)
assert np.allclose(P.sum(axis=1), 1.0)
assert np.all(P >= 0)
print("P is a valid 3x3 row-stochastic matrix.\n")
print("P =\n", P)

def simulate_chain(P, x0, n_steps, rng):
    """Simulate n_steps of a Markov chain with transition matrix P starting at x0."""
    n_states = P.shape[0]
    path = np.empty(n_steps + 1, dtype=int)
    path[0] = x0
    for k in range(n_steps):
        path[k + 1] = rng.choice(n_states, p=P[path[k]])
    return path

rng = np.random.default_rng(42)
path = simulate_chain(P, x0=0, n_steps=30, rng=rng)
print("\nFirst 31 states in the path:")
print(" -> ".join(state_names[s] for s in path))


---
## Part 2 — \( n \)-step transitions: \( P^{(n)} = P^n \)

The \( n \)-step transition probabilities are

$$
p_{ij}^{(n)} = P(X_n = j \mid X_0 = i),
$$

and they are obtained by **matrix powers**: \( P^{(n)} = P^n \), i.e.
\( (P^n)_{ij} = p_{ij}^{(n)} \). This follows from the Chapman–Kolmogorov
identity \( P^{(m+n)} = P^{(m)} P^{(n)} \).

Below we compute \( P^n \) for several \( n \) and compare the row entries
\( (P^n)_{ij} \) to the **empirical frequencies** obtained by simulating many
independent chains for exactly \( n \) steps from a fixed start and counting the
terminal states.


In [ ]:
def matrix_power(P, n):
    """Return P^n for non-negative integer n (numpy has numpy.linalg.matrix_power)."""
    return np.linalg.matrix_power(P, n)

# Compare P^n row 0 to simulated frequencies of X_n starting from state 0
n = 20
n_sim = 40_000
rng = np.random.default_rng(0)

Pn = matrix_power(P, n)
print(f"P^{n} =\n", np.round(Pn, 4))

# Simulate many independent n-step chains from state 0
final_states = np.empty(n_sim, dtype=int)
for s in range(n_sim):
    x = 0
    for _ in range(n):
        x = rng.choice(3, p=P[x])
    final_states[s] = x

freqs = np.array([np.mean(final_states == j) for j in range(3)])
print(f"\nFrom state 0, after n={n} steps:")
print(f"{'state':>10} {'(P^n)_{0j}':>12} {'empirical':>12}")
for j in range(3):
    print(f"{state_names[j]:>10} {Pn[0, j]:>12.4f} {freqs[j]:>12.4f}")

# Max absolute deviation between theory and simulation
mad = np.max(np.abs(Pn[0] - freqs))
print(f"\nmax |theory - empirical| = {mad:.4f}  (n_sim={n_sim})")


---
## Part 3 — Stationary distribution: solve \( \pi^\top P = \pi^\top \)

A probability vector \( \pi \) on \( S \) is **stationary** if

$$
\pi^\top P = \pi^\top,
$$

i.e. \( \pi \) is a **left eigenvector** of \( P \) with eigenvalue 1, normalized
to sum to 1. We solve the linear system \( (P^\top - I)\pi^\top = 0 \) together with
\( \sum_j \pi_j = 1 \) and verify \( \pi P = \pi \). We then check that the
**empirical occupancy** of a long simulated path — the fraction of time spent in each
state — matches \( \pi \) (a preview of the ergodic theorem in Part 4).


In [ ]:
def stationary_distribution(P):
    """Solve pi^T P = pi^T, sum(pi)=1, for a finite irreducible chain."""
    n = P.shape[0]
    # (P^T - I) pi = 0, plus constraint sum(pi)=1. Replace one equation with the constraint.
    A = (P.T - np.eye(n))
    # Replace last row with all-ones (normalization)
    A[-1, :] = 1.0
    b = np.zeros(n)
    b[-1] = 1.0
    pi = np.linalg.solve(A, b)
    # Clip tiny numerical negatives and renormalize
    pi = np.clip(pi, 0, None)
    pi = pi / pi.sum()
    return pi

pi = stationary_distribution(P)
print("Stationary distribution pi =", np.round(pi, 4))

# Verify pi^T P = pi^T
print("pi @ P  =", np.round(pi @ P, 4))
print("max |pi P - pi| =", np.max(np.abs(pi @ P - pi)))

# Empirical occupancy of a long single path
rng = np.random.default_rng(7)
long_path = simulate_chain(P, x0=0, n_steps=200_000, rng=rng)
occupancy = np.array([np.mean(long_path == j) for j in range(3)])
print(f"\n{'state':>10} {'pi_j':>10} {'occupancy':>12}")
for j in range(3):
    print(f"{state_names[j]:>10} {pi[j]:>10.4f} {occupancy[j]:>12.4f}")
print(f"max |pi - occupancy| = {np.max(np.abs(pi - occupancy)):.4f}")


---
## Part 4 — Ergodic theorem: convergence to \( \pi \) from any start

Under irreducibility, aperiodicity, and positive recurrence, a unique stationary
distribution \( \pi \) exists and

$$
\lim_{n\to\infty} p_{ij}^{(n)} = \pi_j, \quad \forall\, i, j.
$$

Moreover, **time averages** converge almost surely:

$$
\frac{1}{n}\sum_{k=1}^{n} f(X_k) \to} \sum_{j\in S} f(j)\,\pi_j,
$$

for any bounded \( f \). This is the **ergodic theorem for Markov chains** — long
sample paths reveal stationary expectations. It is the theoretical basis for MCMC.

Below we (a) watch \( P^n \) converge row-by-row to \( \pi \), and (b) watch the
running occupancy of a single path converge to \( \pi \), starting from each of
the three different states.


In [ ]:
# (a) Row i of P^n should converge to pi as n -> infinity
print("Convergence of rows of P^n to pi:")
print(f"{'n':>6} " + " ".join(f"{s:>10}" for s in state_names) + "   ||   pi")
for n in [1, 2, 5, 10, 20, 50]:
    Pn = matrix_power(P, n)
    row0 = Pn[0]
    print(f"{n:>6} " + " ".join(f"{v:>10.4f}" for v in row0) + f"   ||   " + " ".join(f"{v:.4f}" for v in pi))

# (b) Running occupancy from three different starts
rng = np.random.default_rng(11)
n_steps = 50_000
fig, ax = plt.subplots()
colors = ["tab:red", "tab:green", "tab:blue"]
for i0, c in zip([0, 1, 2], colors):
    path = simulate_chain(P, x0=i0, n_steps=n_steps, rng=rng)
    # running fraction in state 0 (Sunny) as a proxy for f-indicator average
    indicator = (path[1:] == 0).astype(float)
    running = np.cumsum(indicator) / np.arange(1, n_steps + 1)
    ax.plot(np.arange(1, n_steps + 1), running, color=c, lw=0.9,
            label=f"start {state_names[i0]} -> freq of Sunny")
ax.axhline(pi[0], color="black", ls="--", lw=1.5, label=r"$\pi_{\mathrm{Sunny}}$")
ax.set_xscale("log")
ax.set_xlabel("step  $n$")
ax.set_ylabel(r"running fraction in state Sunny")
ax.set_title(r"Ergodicity: running occupancy converges to $\pi$ from any start")
ax.legend()
plt.show()

# Time-average of f = identity (state index) should converge to sum_j j*pi_j
fvals = np.arange(3)
target = np.sum(fvals * pi)
print(f"\nE_pi[f(X)] for f(j)=j : {target:.4f}")
for i0 in [0, 1, 2]:
    path = simulate_chain(P, x0=i0, n_steps=200_000, rng=np.random.default_rng(100 + i0))
    print(f"  start={state_names[i0]:>7}: time-average = {path.mean():.4f}")


---
## Part 5 — Classification of states: recurrent vs transient

A state is **recurrent** if, starting from it, the chain returns to it with
probability 1; otherwise it is **transient**. In a finite irreducible chain every
state is recurrent (in fact positive recurrent). To illustrate transience we use a
chain with an absorbing "sink" state plus some transient states: starting from a
transient state, the chain eventually leaves and never returns.

We estimate the **return probability** and the **mean number of visits** to a state
by simulation. For a transient state the mean number of visits is finite; for a
recurrent state it is infinite (in a long simulation, the count keeps growing).


In [ ]:
# 4-state chain: state 0 is absorbing (recurrent), states 1,2,3 drift toward 0
# Row i gives probabilities of moving 0..3
P_abs = np.array([
    [1.00, 0.00, 0.00, 0.00],   # 0: absorbing
    [0.50, 0.25, 0.25, 0.00],   # 1: transient
    [0.40, 0.10, 0.30, 0.20],   # 2: transient
    [0.30, 0.00, 0.10, 0.60],   # 3: transient (self-loop but eventually leaves)
])
names_abs = ["Absorb0", "T1", "T2", "T3"]
print("P =\n", P_abs)
print("row sums:", P_abs.sum(axis=1))

def mean_visits_vectorized(P, x0, n_rep, horizon, rng):
    """Mean visits to x0 over `horizon` steps for n_rep paths, vectorized.

    State 0 is absorbing: a path starting there stays forever (visits = horizon+1).
    Transient starts absorb quickly, so the per-step loop over still-alive paths
    terminates in a handful of iterations -- no per-path Python loop over `horizon`.
    """
    if x0 == 0:                                   # absorbing: never leaves state 0
        return np.full(n_rep, horizon + 1, dtype=float)
    Pcum = P.cumsum(axis=1)
    x = np.full(n_rep, x0, dtype=int)
    visits = np.ones(n_rep, dtype=float)          # count time 0
    alive = x != 0
    for _ in range(horizon):
        if not alive.any():
            break
        xa = x[alive]
        u = rng.random(size=xa.size)
        x[alive] = (u[:, None] < Pcum[xa]).argmax(axis=1)
        visits[alive] += (x[alive] == x0)
        alive = x != 0
    return visits

rng = np.random.default_rng(5)
H = 200_000
n_rep = 2000
for s in [0, 1, 2, 3]:
    visits = mean_visits_vectorized(P_abs, s, n_rep, H, rng)
    print(f"start={names_abs[s]:>8}: mean #visits over {H} steps = {visits.mean():8.1f}, "
          f"median = {np.median(visits):6.0f}")

# For state 0 (absorbing): every path stays there forever -> visits = H+1.
# For transient states: mean visits is finite (geometric tail).

# Mean absorption time to state 0 from each transient start (theory via linear equations)
# Solve t_i = 1 + sum_{j != 0} P_{ij} t_j  ->  (I - Q) t = 1, Q = P[1:,1:]
Q = P_abs[1:, 1:]
t = np.linalg.solve(np.eye(3) - Q, np.ones(3))
print("\nMean absorption time to state 0 (theory):")
for i, name in enumerate(names_abs[1:]):
    print(f"  from {name}: {t[i]:.3f} steps")

# Check by simulation
rng = np.random.default_rng(9)
for s in [1, 2, 3]:
    times = []
    for _ in range(20_000):
        x = s; k = 0
        while x != 0:
            x = rng.choice(4, p=P_abs[x]); k += 1
        times.append(k)
    print(f"  from {names_abs[s]}: empirical mean absorption time = {np.mean(times):.3f}")


---
## Your turn

- Change the weather chain \( P \) so that state 2 (Rainy) is **absorbing** and
  recompute \( \pi \). What goes wrong with the uniqueness argument? Which
  states are now transient?
- For the original weather chain, plot \( \|P^n_{i\cdot} - \pi\|_1 \) versus
  \( n \) on a log scale for each start \( i \). Does the decay look exponential?
  Estimate the rate from the slope (it is related to the second-largest eigenvalue
  of \( P \)).
- Replace the weather chain with the **deterministic cycle** on \( m=5 \) states
  \( (P(X_{n+1}=i+1\bmod m \mid X_n=i)=1) \). Compute \( P^n \) for
  \( n=1,2,3,4,5,6 \). Why does \( P^n \) **not** converge to the uniform
  distribution even though the uniform distribution is stationary? (Periodicity.)
- For the absorbing chain of Part 5, modify it so state 3 has a 0.05 probability of
  returning to state 1 (making 1 and 3 communicate). Re-classify states and recompute
  the mean absorption times.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. State the **Markov property** in your own words. Write down a 3-state transition matrix \( P \) of your choice and verify it is row-stochastic. Simulate a path of length 100 from state 0 and report the empirical one-step transition counts; do they match the corresponding rows of \( P \)?

2. For a Markov chain with transition matrix \( P \), explain why the \( n \)-step transition matrix is \( P^{(n)} = P^n \). For the weather chain \( P = \begin{pmatrix}0.7&0.2&0.1\\0.3&0.4&0.3\\0.2&0.3&0.5\end{pmatrix} \), compute \( P^{10} \) and \( P^{50} \) and compare each row to the stationary distribution \( \pi \). What do you observe?

3. Define a **stationary distribution** \( \pi \) for a finite Markov chain. Solve \( \pi^\top P = \pi^\top \) for the weather chain above and verify \( \pi P = \pi \). Simulate a single path of length 200,000 and show the empirical occupancy of each state agrees with \( \pi \) to within ~0.005.

4. State the **ergodic theorem for Markov chains**. Using the weather chain, compute \( E_\pi[f(X)] \) for \( f(j)=j \) analytically, then verify it with the time average \( \frac{1}{n}\sum_{k=1}^n f(X_k) \) along a long simulated path from two different starting states. Why does the starting state not matter in the long run?

5. Give the definitions of **recurrent** and **transient** states. Construct a 4-state chain with one absorbing state and three transient states. By simulation, estimate the mean number of visits to each transient state before absorption, and compare with the theoretical mean absorption time obtained by solving \( (I - Q)\mathbf{t} = \mathbf{1} \), where \( Q \) is the submatrix of \( P \) on the transient states.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** The Markov property says
\( P(X_{n+1}=j\mid X_n=i,\,\text{past}) = P(X_{n+1}=j\mid X_n=i) = p_{ij} \):
the next state depends only on the current state, not the earlier history. A
row-stochastic matrix has non-negative entries and each row sums to 1.

```python
import numpy as np
rng = np.random.default_rng(0)
P = np.array([[0.6,0.3,0.1],[0.2,0.5,0.3],[0.1,0.4,0.5]])
assert np.allclose(P.sum(axis=1), 1.0)
path = [0]
for _ in range(100):
    path.append(rng.choice(3, p=P[path[-1]]))
path = np.array(path)
# Empirical 1-step counts from state 0
from collections import Counter
c = Counter()
for a, b in zip(path[:-1], path[1:]):
    c[a] += [0,0,0]  # placeholder
# Simpler: count transitions
trans = np.zeros((3,3))
for a, b in zip(path[:-1], path[1:]):
    trans[a, b] += 1
emp = trans / trans.sum(axis=1, keepdims=True)
print("P  =\n", P)
print("emp=\n", np.round(emp, 3))
```
With only ~100 steps the empirical rows are noisy; they sharpen as the path length
grows (a side-effect of the ergodic theorem applied to the pair \( (X_k, X_{k+1}) \)).

**2.** The Chapman–Kolmogorov identity \( P^{(m+n)} = P^{(m)}P^{(n)} \) implies
\( P^{(n)} = P^n \) by induction. Computing in Python:

```python
import numpy as np
P = np.array([[0.7,0.2,0.1],[0.3,0.4,0.3],[0.2,0.3,0.5]])
P10 = np.linalg.matrix_power(P, 10)
P50 = np.linalg.matrix_power(P, 50)
# Stationary distribution
A = (P.T - np.eye(3)); A[-1] = 1.0; b = np.array([0,0,1.0])
pi = np.linalg.solve(A, b); pi = np.clip(pi,0,None); pi /= pi.sum()
print("P^10 =\n", np.round(P10, 4))
print("P^50 =\n", np.round(P50, 4))
print("pi   =", np.round(pi, 4))
```
Each row of \( P^{50} \) is essentially equal to \( \pi \) — the chain has
mixed. \( P^{10} \) still shows a mild dependence on the starting state.

**3.** \( \pi \) is stationary if \( \pi^\top P = \pi^\top \) and
\( \sum_j \pi_j = 1 \). Solving:

```python
import numpy as np
P = np.array([[0.7,0.2,0.1],[0.3,0.4,0.3],[0.2,0.3,0.5]])
A = (P.T - np.eye(3)); A[-1] = 1.0; b = np.array([0,0,1.0])
pi = np.clip(np.linalg.solve(A, b), 0, None); pi /= pi.sum()
print("pi     =", pi, " check pi @ P =", pi @ P)
# Long path
rng = np.random.default_rng(7); x = 0; N = 200_000; counts = np.zeros(3)
for _ in range(N):
    counts[x] += 1; x = rng.choice(3, p=P[x])
occ = counts / counts.sum()
print("occupancy =", np.round(occ, 4), " max|pi-occ| =", np.max(np.abs(pi-occ)))
```
The max deviation should be of order \( 1/\sqrt{N} \approx 0.002 \), well under
0.005.

**4.** Ergodic theorem: if the chain is irreducible, aperiodic, and positive
recurrent, then \( \frac{1}{n}\sum_{k=1}^n f(X_k) \to \sum_j f(j)\pi_j \)
almost surely, regardless of \( X_0 \). For \( f(j)=j \):

```python
import numpy as np
P = np.array([[0.7,0.2,0.1],[0.3,0.4,0.3],[0.2,0.3,0.5]])
A = (P.T - np.eye(3)); A[-1] = 1.0; b = np.array([0,0,1.0])
pi = np.clip(np.linalg.solve(A, b), 0, None); pi /= pi.sum()
Ef = np.sum(np.arange(3) * pi)
print("E_pi[f] =", Ef)
for x0 in [0, 1, 2]:
    rng = np.random.default_rng(100 + x0); x = x0; s = 0.0; N = 300_000
    for _ in range(N):
        x = rng.choice(3, p=P[x]); s += x
    print(f" start={x0}: time-avg = {s/N:.4f}")
```
Both starts give \( \approx E_\pi[f] \). The starting state does not matter
because \( p_{ij}^{(n)}\to\pi_j \), so the chain forgets its initial condition
after a mixing time.

**5.** State \( i \) is **recurrent** if \( P(\text{ever return to } i \mid X_0=i)=1 \),
otherwise **transient**. In a finite chain, states that can reach an absorbing class
but cannot be returned to are transient.

```python
import numpy as np
P = np.array([[1.0,0,0,0],[0.5,0.25,0.25,0],[0.4,0.1,0.3,0.2],[0.3,0,0.1,0.6]])
Q = P[1:, 1:]                      # transient block
t = np.linalg.solve(np.eye(3) - Q, np.ones(3))
print("mean absorption time:", t)
# Mean number of visits to transient state i before absorption is ((I-Q)^{-1})_{ii}
mean_visits = np.linalg.solve(np.eye(3) - Q, np.eye(3)).diagonal()
print("mean visits to each transient state:", mean_visits)
# Simulation check
rng = np.random.default_rng(9)
for s in [1, 2, 3]:
    visits = 0; runs = 20000
    for _ in range(runs):
        x = s; v = 0
        while x != 0:
            if x == s: v += 1
            x = rng.choice(4, p=P[x])
        visits += v
    print(f" state {s}: empirical mean visits = {visits/runs:.3f}")
```
The empirical mean visits match the diagonal of \( (I-Q)^{-1} \), and the mean
absorption time matches \( \mathbf{t} = (I-Q)^{-1}\mathbf{1} \).


</details>
